# 1. Environment Setup & Library Imports

To ensure our pipeline is completely reproducible and easy for the Zindi code review team to execute, we import and initialize all required libraries in this first cell.

### Libraries & Tools Used:
* **`catboost`**: Our primary gradient boosting model. It is chosen because it handles categorical variables (like job types and education levels) exceptionally well and is highly stable.
* **`pandas` & `numpy`**: For structural data manipulation and linear algebra operations.
* **`StratifiedKFold`**: To split our training data into 10 distinct, balanced folds. This guarantees our validation sets mirror the exact class distribution of the overall dataset.
* **`mean_absolute_error` (MAE)**: The metric used to calculate our local Out-of-Fold (OOF) score and find the optimal decision threshold.

### Reproducibility Notice:
To prevent package update discrepancies, exact versions are documented in our accompanying `requirements.txt` file. All stochastic operations in subsequent training loops utilize fixed random seeds (`42`, `2024`, and `7`) to guarantee our leaderboard score of **Rank 7** is perfectly reproducible.

In [ ]:
!pip install catboost -q

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

# 2. Data Loading & Environment Profile

In this cell, we load the official competition datasets directly into our workspace.

### Environment & Data Path:
* **Storage Location:** Google Drive (`/content/drive/MyDrive/zindi/`)
* **Environment Profile:** Google Colab (CPU runtime)
* **Datasets Loaded:**
  * `Train.csv`: Contains the demographic survey responses and the target variable (`bank_account`).
  * `Test.csv`: Contains the demographic features for the testing partition. Our goal is to predict the missing `bank_account` labels for this set.

*Note: In accordance with competition rules, only the official datasets provided on the Zindi platform are utilized. No external data has been introduced.*

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/zindi/Train.csv')
test = pd.read_csv('/content/drive/MyDrive/zindi/Test.csv')

# 3. Feature Engineering & Preprocessing Pipeline

This section implements our core feature engineering strategies. It temporarily merges the training and testing sets to guarantee consistent formatting, extracts key demographic indicators, builds domain-specific interaction terms, and applies frequency and target encoding to categorical variables.

### Key Engineering Steps:
1. **Dataset Unification**: Combines the train and test sets into a single DataFrame (`combined`) to prevent feature shape mismatches during downstream transformation.
2. **Boolean Indicators**: Simplifies key demographic variables (e.g., location, cellphone access, gender, household role) into binary $0/1$ flags.
3. **Ordinal Education Mapping**: Maps text descriptions of education level into an ascending numeric scale representing progressive educational attainment.
4. **Binning & Interaction Features**: Groups age and household sizes into logical intervals and creates cross-feature interactions (like cellphone ownership paired with urban residency) to capture non-linear relationships.
5. **Frequency & Target Encoding**:
   * **Frequency Encoding**: Represents the prevalence of categorical segments in the overall population context.
   * **Target Encoding**: Uses the historical likelihood of bank account ownership (`target` mean) per category as a powerful direct numerical signal.

In [ ]:
# FEATURE ENGINEERING & PIPELINE ALIGNMENT

# Temporarily initialize target in test set as NaN before concatenation
# so we can cleanly separate them back after completing transformations
test['bank_account'] = np.nan
combined = pd.concat([train, test], axis=0).reset_index(drop=True)

# Define Binary Indicators (0 or 1) for strong demographic signals
combined['is_urban'] = (combined['location_type'] == 'Urban').astype(int)
combined['has_cellphone'] = (combined['cellphone_access'] == 'Yes').astype(int)
combined['is_male'] = (combined['gender_of_respondent'] == 'Male').astype(int)
combined['is_head'] = (combined['relationship_with_head'] == 'Head of Household').astype(int)

# Group formal employment sectors together as a single feature
combined['is_formal'] = combined['job_type'].isin(
    ['Formally employed Government', 'Formally employed Private']
).astype(int)

# Map categorical education levels into an ordinal scale (0 to 4)
edu_order = {
    'No formal education': 0, 'Other/Dont know/RTA': 0,
    'Primary education': 1, 'Secondary education': 2,
    'Vocational/Specialised training': 3, 'Tertiary education': 4
}
combined['education_rank'] = combined['education_level'].map(edu_order).fillna(0)

# Bin numerical features (Age & Household size) into logical age/size categories
combined['age_group'] = pd.cut(
    combined['age_of_respondent'],
    bins=[0, 25, 35, 50, 65, 100],
    labels=[0, 1, 2, 3, 4]
).astype(int)

combined['hh_size_bin'] = pd.cut(
    combined['household_size'],
    bins=[0, 2, 4, 6, 100],
    labels=[0, 1, 2, 3]
).astype(int)

# Real-world constraint: Ratio of household size to the age of the respondent
# This represents the structural financial pressure/dependency ratio on the individual
combined['household_pressure'] = combined['household_size'] / (combined['age_of_respondent'] + 1)

# Interaction Features: Capturing cumulative socio-economic impact
combined['cellphone_x_urban'] = combined['has_cellphone'] * combined['is_urban']
combined['cellphone_x_edu'] = combined['has_cellphone'] * combined['education_rank']
combined['age_x_edu'] = combined['age_of_respondent'] * combined['education_rank']
combined['formal_x_urban'] = combined['is_formal'] * combined['is_urban']

# Frequency Encoding: Convert text labels to their global dataset frequency (relative popularity)
for col in ['job_type', 'education_level', 'country', 'relationship_with_head', 'marital_status']:
    freq = combined[col].value_counts(normalize=True)
    combined[col + '_freq'] = combined[col].map(freq)

# Identify the rows belonging to the training set (to avoid using test NaN values for target means)
train_mask = combined['bank_account'].notna()

# Convert target labels to temporary binary values for statistical mean calculations
combined['target'] = combined['bank_account'].map({'Yes': 1, 'No': 0})

# Global Target Encoding: Maps the average positive ('Yes') rate for each category
# based strictly on the training partition
for col in ['country', 'job_type', 'education_level', 'relationship_with_head', 'marital_status']:
    means = combined.loc[train_mask].groupby(col)['target'].mean()
    combined[col + '_target_enc'] = combined[col].map(means)

# 4. Feature Selection & Dataset Partitioning

In this phase, we isolate our final feature set from the helper columns used during preprocessing. We separate the combined dataset back into distinct training and testing matrices based on our training mask, ready to be ingested by the CatBoost classifier.

### Key Operations:
1. **Dumping Non-Predictive Columns**:
   * Drops identifiers like `uniqueid` which provide no generalizable signals.
   * Drops raw categorical string columns (e.g., `job_type`, `country`) since we have already created numerical frequency, ordinal, and target-encoded representations for them.
   * Excludes raw labels (`bank_account` and the temporary helper column `target`).
2. **Feature Isolation**: Gathers all engineered features, numerical scales, and encoded features into a final list (`feature_cols`).
3. **Array Splitting**:
   * **`X` (Train Features)**: Subset of rows where the target label originally existed.
   * **`y` (Train Target)**: The binary mapping ($0/1$) of target classes for training.
   * **`X_test` (Test Features)**: Evaluation set matching the unlabeled rows destined for final leaderboard prediction.

In [ ]:
# FEATURE ISOLATION & TRAIN/TEST PARTITIONING

# Define raw categorical strings, unique IDs, and target columns to exclude.
# This prevents data leakage and ensures the model only trains on numerical features.
drop_cols = [
    'uniqueid', 'country', 'location_type', 'cellphone_access',
    'gender_of_respondent', 'relationship_with_head', 'marital_status',
    'education_level', 'job_type', 'bank_account', 'target'
]

# Dynamically select columns that are not included in the drop list
feature_cols = [col for col in combined.columns if col not in drop_cols]

# Partition back to the final training features and labels using the mask
X = combined[train_mask][feature_cols]
y = combined[train_mask]['target']

# Partition the unlabeled test features using the inverse of the training mask
X_test = combined[~train_mask][feature_cols]

# Visual validation of dataset dimensions before feeding into the modeling pipeline
print(f"X: {X.shape} | X_test: {X_test.shape} | Features: {len(feature_cols)}")

X: (23524, 26) | X_test: (10086, 26) | Features: 26


# 5. Triple-Seed CatBoost Ensemble & Out-of-Fold (OOF) Training

This section defines the engine of our solution: a robust, multi-seed cross-validated ensemble. Instead of relying on a single train-test split (which can introduce high variance), this code trains exactly 30 model variations (10-Fold Stratified CV across 3 distinct random seeds).

### Key Architectural Concepts:
1. **Stratified 10-Fold Cross-Validation**: Divides the training set into 10 folds, keeping the ratio of the target classes ('Yes' vs. 'No') consistent across each fold.
2. **Triple-Seed Ensembling**: Employs seeds `42`, `2024`, and `7`. By repeating the cross-validation with different shuffle states, we smooth out any random variations in fold generation.
3. **Out-of-Fold (OOF) Predictions**: Generates model predictions on validation folds that the model never saw during training. The combined OOF predictions provide a highly reliable benchmark of local model accuracy.
4. **Ensemble Averaging**: Blends the prediction probabilities from all 30 trained models (10 folds × 3 seeds) for final test inference to maximize generalized performance on the private leaderboard.

In [ ]:
# TRIPLE-SEED CATBOOST ENSEMBLING PIPELINE

# Define three stable, distinct random seeds to capture variance and prevent overfitting
seeds = [42, 2024, 7]

# Initialize arrays to hold out-of-fold validation predictions and final test predictions
oof_all = np.zeros(len(X))
test_all = np.zeros(len(X_test))

# Iterate through each independent seed loop
for seed in seeds:
    # Set up Stratified 10-Fold Cross Validation for the current seed
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))
    seed_test = np.zeros(len(X_test))

    # Loop through each of the 10 folds
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        # Partition data into active training and validation sets for this fold
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Initialize the CatBoost Classifier with optimized parameters
        model = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.02,
            depth=6,
            l2_leaf_reg=12,
            task_type='CPU',
            eval_metric='Logloss',
            early_stopping_rounds=200, # Stops early if validation loss plateaus
            verbose=0                  # Keeps the console clean during the 30-step loop
        )

        # Train the model with early stopping active on the validation set
        model.fit(X_train, y_train, eval_set=(X_val, y_val))

        # Save Out-of-Fold (OOF) predictions for this fold (probability of class 1)
        seed_oof[val_idx] = model.predict_proba(X_val)[:, 1]

        # Predict test probabilities and accumulate the scaled 1/10th fold contribution
        seed_test += model.predict_proba(X_test)[:, 1] / 10

    # Average the seed validation predictions and add them to the global ensemble array
    oof_all += seed_oof / len(seeds)

    # Average the seed test predictions and add them to the global ensemble array
    test_all += seed_test / len(seeds)

    print(f"Seed {seed} done.")

Seed 42 done.
Seed 2024 done.
Seed 7 done.


# 6. Decision Threshold Optimization

By default, classification models use a standard probability threshold of $0.5$ to distinguish between classes. However, in highly imbalanced datasets, this arbitrary threshold is rarely optimal. This section sweeps across a range of potential classification thresholds to identify the boundary that minimizes our Out-of-Fold (OOF) Mean Absolute Error (MAE).

### Key Concepts:
1. **The Metric (Mean Absolute Error)**: Because the target variable and predictions are binary ($0$ or $1$), the Mean Absolute Error is mathematically equivalent to the classification error rate (i.e., $1 - \text{Accuracy}$). Minimizing MAE directly maximizes our prediction accuracy.
2. **Threshold Grid Search**: We sweep from a low probability boundary of $0.20$ to a high of $0.65$ using small, precise steps of $0.01$.
3. **Probability Binary Mapping**: For each prospective threshold $t$, any sample with a combined ensemble probability greater than or equal to $t$ is mapped to $1$ ("Yes"), and all others to $0$ ("No").
4. **Validation Grounding**: Because we are optimizing the threshold on the complete Out-of-Fold (OOF) array (`oof_all`), we avoid overfitting to a single training subset and select a decision boundary that translates directly to unseen leaderboard data.

In [ ]:
# DECISION THRESHOLD OPTIMIZATION


# Generate a grid of candidate thresholds from 0.20 to 0.65 with step increments of 0.01
thresholds = np.arange(0.20, 0.65, 0.01)

# Initialize tracking variables with conservative baseline values
best_thresh, best_mae = 0.5, 999.0

# Evaluate model error across each candidate threshold in the grid
for t in thresholds:
    # Convert prediction probabilities to binary predictions (0 or 1) using the threshold 't'
    binary_predictions = (oof_all >= t).astype(int)

    # Calculate the Mean Absolute Error between true target values and binary predictions
    mae = mean_absolute_error(y, binary_predictions)

    # If the current threshold yields a lower MAE (error rate), update our optimal records
    if mae < best_mae:
        best_mae = mae
        best_thresh = t

# Print the optimized decision threshold and its corresponding Out-of-Fold performance
print(f"\nBest threshold: {best_thresh:.2f} | OOF MAE: {best_mae:.5f}")


Best threshold: 0.48 | OOF MAE: 0.11155

Best threshold: 0.48 | OOF MAE: 0.11155


# 7. Final Inference & Zindi Submission Generation

In this final phase of the pipeline, we apply the optimized decision threshold to our accumulated ensemble probabilities. We then format the predictions precisely to match the sample submission specifications required by the Zindi evaluation server and save the final file.

### Key Operations:
1. **Adaptive Threshold Application**: Uses our grid-searched optimal decision threshold (`best_thresh`) instead of a generic $0.5$ boundary to turn ensemble probabilities into final $0$ (No) and $1$ (Yes) labels.
2. **Key ID Formatting**: Concatenates `uniqueid` and `country` with an `' x '` delimiter (e.g., `uniqueid_6056 x Kenya`) to generate unique keys that align perfectly with Zindi's expected index columns.
3. **Submission Exporting**: Saves the results as `submission_v2.csv` without row indices.
4. **Statistical Diagnostics**: Calculates the percentage and count of positive predictions ("Yes" rate) to verify that our predictions align logically with the actual regional distribution of bank account ownership.

In [ ]:

# FINAL INFERENCE & SUBMISSION GENERATION


# Binarize average ensemble test predictions using our optimized decision threshold
final_labels = (test_all >= best_thresh).astype(int)

# Package predictions into Zindi's strict target formatting structure
submission = pd.DataFrame({
    'unique_id': test['uniqueid'] + ' x ' + test['country'],
    'bank_account': final_labels
})

# Save the predictions locally as 'submission_v2.csv' without index headers
submission.to_csv('submission_v2.csv', index=False)

# Print diagnostic distribution stats to verify predicted target ratio consistency
print(f"\nPredicted 1 (Yes - Account Holder): {final_labels.sum()} ({final_labels.mean():.1%})")
print(f"Predicted 0 (No - No Account):      {(1-final_labels).sum()} ({(1-final_labels).mean():.1%})")

# Visual verification of the head rows before submitting the file
print("\nFirst 5 rows of submission:")
print(submission.head())


Predicted 1 (Yes - Account Holder): 782 (7.8%)
Predicted 0 (No - No Account):      9304 (92.2%)

First 5 rows of submission:
               unique_id  bank_account
0  uniqueid_6056 x Kenya             1
1  uniqueid_6060 x Kenya             1
2  uniqueid_6065 x Kenya             0
3  uniqueid_6072 x Kenya             0
4  uniqueid_6073 x Kenya             0

Predicted 1 (Yes - Account Holder): 782 (7.8%)
Predicted 0 (No - No Account):      9304 (92.2%)

First 5 rows of submission:
               unique_id  bank_account
0  uniqueid_6056 x Kenya             1
1  uniqueid_6060 x Kenya             1
2  uniqueid_6065 x Kenya             0
3  uniqueid_6072 x Kenya             0
4  uniqueid_6073 x Kenya             0


In [ ]:
from google.colab import files
files.download('submission_v2.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>